In [1]:
!pip install -q pandas numpy scikit-learn tensorflow

In [2]:
import pandas as pd
import numpy as np

def load_and_label(filename, posture, activity):
    df = pd.read_csv(filename)
    df["posture"] = posture
    df["activity"] = activity
    return df

# List ALL your files (add new people here too)
files = [
    ("good_posture_jogging.csv", "good", "jogging"),
    ("bad_posture_jogging.csv",  "bad",  "jogging"),
    ("good_posture_walking.csv", "good", "walking"),
    ("bad_posture_walking.csv",  "bad",  "walking"),
    ("good_posture_sitting.csv", "good", "sitting"),
    ("bad_posture_sitting.csv",  "bad",  "sitting"),
    # person2_xxx.csv, person3_xxx.csv, ...
]

fs = 50.0
window_size = int(2 * fs)
step_size = int(1 * fs)

feature_rows = []

for fname, posture, activity in files:
    df = pd.read_csv(fname)
    df = df.sort_values("timestamp").reset_index(drop=True)

    for start in range(0, len(df) - window_size + 1, step_size):
        end = start + window_size
        w = df.iloc[start:end]

        dt = (w["timestamp"].iloc[-1] - w["timestamp"].iloc[0]) / 1000.0
        if dt < 1.5 or dt > 2.5:
            continue

        feats = {}
        for col in ["ax", "ay", "az", "gx", "gy", "gz"]:
            feats[f"{col}_mean"] = w[col].mean()
            feats[f"{col}_std"] = w[col].std()

        acc_mag = np.sqrt(w["ax"]**2 + w["ay"]**2 + w["az"]**2)
        gyro_mag = np.sqrt(w["gx"]**2 + w["gy"]**2 + w["gz"]**2)

        feats["acc_mag_mean"] = acc_mag.mean()
        feats["acc_mag_std"] = acc_mag.std()
        feats["gyro_mag_mean"] = gyro_mag.mean()
        feats["gyro_mag_std"] = gyro_mag.std()

        feats["posture"] = posture
        feats["activity"] = activity

        feature_rows.append(feats)

feat_df = pd.DataFrame(feature_rows)
print("Windows:", len(feat_df))
print(feat_df["posture"].value_counts())
feat_df.to_csv("imu_features_all.csv", index=False)


Windows: 1191
posture
good    698
bad     493
Name: count, dtype: int64


In [3]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feat_df = pd.read_csv("imu_features_all.csv")
feat_df["posture_label"] = (feat_df["posture"] == "bad").astype(np.int32)

feature_cols = [c for c in feat_df.columns if c not in ["posture", "activity", "posture_label"]]
X = feat_df[feature_cols].values
y = feat_df["posture_label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

input_dim = X_train_s.shape[1]
print("Input dim:", input_dim)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation="relu", input_shape=(input_dim,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy"])

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_test_s, y_test),
    epochs=40,
    batch_size=32,
    verbose=1
)

loss, acc = model.evaluate(X_test_s, y_test, verbose=0)
print("Test accuracy:", acc)

# Save scaler
np.savez("scaler_params.npz",
         mean=scaler.mean_.astype(np.float32),
         scale=scaler.scale_.astype(np.float32))

# Convert to TFLite
@tf.function(input_signature=[tf.TensorSpec(shape=[1, input_dim], dtype=tf.float32)])
def model_predict(x):
    return model(x, training=False)

converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [model_predict.get_concrete_function()]
)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()
with open("posture_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite size:", len(tflite_model))


Input dim: 16


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/40
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.6175 - loss: 0.6267 - val_accuracy: 0.8243 - val_loss: 0.4944
Epoch 2/40
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8512 - loss: 0.4309 - val_accuracy: 0.8703 - val_loss: 0.3564
Epoch 3/40
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8775 - loss: 0.3054 - val_accuracy: 0.9038 - val_loss: 0.2708
Epoch 4/40
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9378 - loss: 0.2264 - val_accuracy: 0.9414 - val_loss: 0.2139
Epoch 5/40
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9574 - loss: 0.1573 - val_accuracy: 0.9456 - val_loss: 0.1731
Epoch 6/40
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9626 - loss: 0.1432 - val_accuracy: 0.9498 - val_loss: 0.1424
Epoch 7/40
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9757 - loss: 0.1065 - val_accuracy: 0.9540 - val_loss: 0.1206
Epoch 8/40
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9775 - loss: 0.0913 - val_accuracy: 0.9623 - val

TFLite size: 6160


In [4]:
# Convert TFLite model to C header + scaler to C header

import pathlib

# 1. Convert TFLite model to C array
path = pathlib.Path("posture_model.tflite")
data = path.read_bytes()

with open("posture_model_data.h", "w") as f:
    f.write('#ifndef POSTURE_MODEL_DATA_H\n')
    f.write('#define POSTURE_MODEL_DATA_H\n\n')
    f.write('#include <cstdint>\n\n')

    f.write('alignas(8) const unsigned char posture_model_tflite[] = {\n')
    for i, b in enumerate(data):
        if i % 12 == 0:
            f.write('  ')
        f.write(f'0x{b:02x}')
        if i < len(data) - 1:
            f.write(', ')
        if i % 12 == 11:
            f.write('\n')
    f.write('\n};\n\n')
    f.write(f'const unsigned int posture_model_tflite_len = {len(data)};\n\n')
    f.write('#endif\n')

print(f"Generated posture_model_data.h ({len(data)} bytes)")

# 2. Convert scaler params to C header
scaler_data = np.load("scaler_params.npz")
mean = scaler_data["mean"]
scale = scaler_data["scale"]

with open("scaler_data.h", "w") as f:
    f.write("#ifndef SCALER_DATA_H\n")
    f.write("#define SCALER_DATA_H\n\n")
    f.write(f"const int FEATURE_DIM = {len(mean)};\n\n")
    f.write("const float scaler_mean[] = {")
    f.write(", ".join([f"{m:.6f}f" for m in mean]))
    f.write("};\n\n")
    f.write("const float scaler_scale[] = {")
    f.write(", ".join([f"{s:.6f}f" for s in scale]))
    f.write("};\n\n")
    f.write("#endif\n")

print("Generated scaler_data.h")

# 3. Download both header files
from google.colab import files
files.download("posture_model_data.h")
files.download("scaler_data.h")



Generated posture_model_data.h (6160 bytes)
Generated scaler_data.h


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>